In [1]:
%load_ext autoreload
%autoreload 2
%reset -f

# Imports

In [2]:
from locallib.picarrodb import *
from locallib.query import *
from locallib.box import *
from locallib.query import *

import pandas as pd
import geopandas as gpd
import contextily as ctx
import pandas as pd
import os

import sys
import os

from shapely import wkt
from shapely.ops import unary_union
import matplotlib.pyplot as plt
import h3

os.chdir('/home/sandbox/personal-repos/DA-3507/dump')
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../lib')))
from custom_pandas import *

/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/geopandas/_compat.py:154: UserWarning: The Shapely GEOS version (3.10.3-CAPI-1.16.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.1-CAPI-1.16.0). Conversions between both will be slow.
  set_use_pygeos()


EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully


## Get the surveys

In [3]:
a = get_reports('Cadent',years = [2026]).execute([EU2_Conn])

In [4]:
query = f"""SELECT RA.Shape.STAsText() AS SurveyArea,
    RA.ReportId
FROM ReportArea RA
WHERE RA.ReportId IN (SELECT ReportId FROM #TempReport)"""
report_bc = a.head(1).copy()
report_bc

,CustomerName,ReportName,ReportId,ReportTitle,Label,ReportDate,BoundaryName,BoundaryType,ReportAssetLengthKm,ReportPercentCoverageAssets,...,DistributionPipeCoveredKm,DistributionPipePercentCovered,ServicePipeKm,ServicePipeCoveredKm,AreaKM2,AreaCoveredKM2,ReportYear,ReportMonth,ReportWeek,ReportQuarter
0,Cadent,CR-6704DC,6704DCF9-CD46-2467-12B8-3A20F48D4B77,B26 - 3499 - WM South - Wyre Forest - Aggborou...,Final Checkbox,2026-05-01 07:11:05.080,B26 - 3499 - WM South - Wyre Forest - Aggborou...,--Select--,34.57349,0.993357,...,34.100381,0.994523,0.285319,0.243428,2.780409,1.893048,2026,5,18,2


In [5]:
report_bc.db.set_query(query_SurveyH3Aggregation_byReport(report_table = '#TempReport'))
data =report_bc.db.execute([EU2_Conn], source_col = 'ReportId', temp_table_name = '#TempReport')
report_bc.db.set_query(query)
boundary = report_bc.db.execute([EU2_Conn], source_col = 'ReportId', temp_table_name = '#TempReport')

In [6]:

data_gdf = gpd.GeoDataFrame(
    data.copy(),
    geometry=data["Breadcrumb"].apply(wkt.loads),
    crs="EPSG:4326"
)
boundary_gdf = gpd.GeoDataFrame(
    boundary.copy(),
    geometry=boundary["SurveyArea"].apply(wkt.loads),
    crs="EPSG:4326"
)

In [7]:
merged = pd.merge(data, boundary, on = 'ReportId', how = 'left')
merged.drop(columns = ['ReportId'], inplace = True)

In [8]:
def h3_cell_polygon(cell):
    # Returns a shapely Polygon for an h3 cell index (in degrees)
    boundary = h3.cell_to_boundary(cell)
    # Invert lat and long (swap their positions in each tuple)
    boundary = [(lat, lon) for lon, lat in boundary]
    return Polygon(boundary)

In [9]:
h3_resolution = 11
utm_crs = boundary_gdf.estimate_utm_crs()
boundary_gdf_proj = boundary_gdf.to_crs(utm_crs)
boundary_gdf_proj["geometry"] = boundary_gdf_proj["geometry"].buffer(200)  # buffer by 100 meters
# Project back to EPSG:4326 for further processing
boundary_gdf_buffered = boundary_gdf_proj.to_crs(epsg=4326)
boundary_geom = boundary_gdf_buffered.geometry.values[0]

 # H3 v4+: polygon_to_cells expects LatLngPoly/LatLngMultiPoly, not a GeoJSON dict.
            # geo_to_cells accepts Shapely (via __geo_interface__) or a GeoJSON dict.
h3_cells = h3.geo_to_cells(boundary_geom, h3_resolution)
h3_cells_list = list(h3_cells)

# Create polygons for each H3 cell
h3_polys = [h3_cell_polygon(cell) for cell in h3_cells_list]
h3_gdf = gpd.GeoDataFrame({'h3_cell': h3_cells_list, 'geometry': h3_polys}, crs='EPSG:4326')
h3_gdf = h3_gdf.to_crs(utm_crs)
# Project h3_gdf to a projected CRS (e.g., Web Mercator) before buffering
h3_gdf_proj = h3_gdf.to_crs(utm_crs)
h3_gdf_proj['offset'] = h3_gdf_proj['geometry'].buffer(-2)  # 2 meters in projected CRS (meters)
data_gdf = data_gdf.to_crs(utm_crs)


In [10]:
polygon_lines = h3_gdf_proj.offset.boundary
polygon_lines_gdf = gpd.GeoDataFrame({"geometry": polygon_lines}, crs=utm_crs)

# Intersect the lines of the polygons with the breadcrumbs
intersection_gdf = gpd.overlay(polygon_lines_gdf, data_gdf, how='intersection', keep_geom_type=False)
intersection_gdf = intersection_gdf.explode()
intersection_gdf.to_crs(utm_crs)

/tmp/ipykernel_5832/2208351934.py:6: FutureWarning: Currently, index_parts defaults to True, but in the future, it will default to False to be consistent with Pandas. Use `index_parts=True` to keep the current behavior and True/False to silence the warning.
  intersection_gdf = intersection_gdf.explode()


ReportId  \
0    0  6704DCF9-CD46-2467-12B8-3A20F48D4B77   
     1  6704DCF9-CD46-2467-12B8-3A20F48D4B77   
1    0  6704DCF9-CD46-2467-12B8-3A20F48D4B77   
     1  6704DCF9-CD46-2467-12B8-3A20F48D4B77   
2    0  6704DCF9-CD46-2467-12B8-3A20F48D4B77   
...                                      ...   
2342 3  6704DCF9-CD46-2467-12B8-3A20F48D4B77   
2343 0  6704DCF9-CD46-2467-12B8-3A20F48D4B77   
     1  6704DCF9-CD46-2467-12B8-3A20F48D4B77   
2344 0  6704DCF9-CD46-2467-12B8-3A20F48D4B77   
     1  6704DCF9-CD46-2467-12B8-3A20F48D4B77   

                                    SurveyId  \
0    0  89609852-C748-23A9-64B1-3A20E92FED13   
     1  89609852-C748-23A9-64B1-3A20E92FED13   
1    0  89609852-C748-23A9-64B1-3A20E92FED13   
     1  89609852-C748-23A9-64B1-3A20E92FED13   
2    0  89609852-C748-23A9-64B1-3A20E92FED13   
...                                      ...   
2342 3  F9495428-9F5C-F053-FF7C-3A20F35B5950   
2343 0  F9495428-9F5C-F053-FF7C-3A20F35B5950   
     1  F9495428-9F5C-F053-FF7C-3A20F35B5950   
2344 0  F9495428-9F5C-F053-FF7C-3A20F35B5950   
     1  F9495428-9F5C-F053-FF7C-3A20F35B5950   

                                               Breadcrumb  \
0    0  MULTILINESTRING ((-2.2266278948597491 52.37678...   
     1  MULTILINESTRING ((-2.2266278948597491 52.37678...   
1    0  MULTILINESTRING ((-2.2266278948597491 52.37678...   
     1  MULTILINESTRING ((-2.2266278948597491 52.37678...   
2    0  MULTILINESTRING ((-2.2266278948597491 52.37678...   
...                                                   ...   
2342 3  MULTILINESTRING ((-2.2474755173994585 52.37650...   
2343 0  MULTILINESTRING ((-2.2474755173994585 52.37650...   
     1  MULTILINESTRING ((-2.2474755173994585 52.37650...   
2344 0  MULTILINESTRING ((-2.2474755173994585 52.37650...   
     1  MULTILINESTRING ((-2.2474755173994585 52.37650...   

                              geometry  
0    0  POINT (551936.640 5802409.669)  
     1  POINT (551936.650 5802409.664)  
1    0  POINT (552138.751 5802055.123)  
     1  POINT (552141.164 5802057.041)  
2    0  POINT (552136.362 5802144.805)  
...                                ...  
2342 3  POINT (551191.029 5803179.034)  
2343 0  POINT (551322.111 5803290.993)  
     1  POINT (551365.210 5803291.102)  
2344 0  POINT (551392.559 5803200.950)  
     1  POINT (551376.308 5803199.069)  

[10551 rows x 4 columns]

In [11]:
intersection_gdf.reset_index(drop=True, inplace=True)
pl = h3_gdf.boundary


# Only keep point geometries in intersection_gdf (if other types present)
points_gdf = intersection_gdf[intersection_gdf.geometry.type == 'Point']

# Spatial join: which points fall on a line
joined = gpd.sjoin(points_gdf, pl, how='inner', predicate='intersects')

count = len(joined)
print("Number of points inside pl (via sjoin):", count)

ValueError: 'right_df' should be GeoDataFrame, got <class 'geopandas.geoseries.GeoSeries'>

In [ ]:
merged.BreadcrumbCounts.set_H3Resolution(11)
merged.BreadcrumbCounts.set_counts_only(False)
a = merged.BreadcrumbCounts.calculate('Breadcrumb','SurveyArea')

In [ ]:
a